# Notebook 04 — Reconcile Confirmed Overlaps (No Image Deletion)

**Purpose:** After manual review of dHash candidates, produce a reproducible **cleaned candidate** train/validation/test manifest.

We do **not** delete, move, crop, or relabel original images. We preserve all **751 original publisher test images**. If a publisher-training image is verified as a duplicate/near-identical frame of a test image, exclude that training-origin image from both training and validation so that the test observation is not memorized during training. Put verified groups of similar training-origin images entirely within training or validation.

**The input review CSV is critical.** Do not assume two vehicles are identical merely because their make/model folders match. A confirmed match means the same physical vehicle in identical or near-identical frames. Screening finds candidates, not all potential overlaps.

**This notebook requires completed human judgments and halts if any verdict is still `unreviewed` or `uncertain`.** Final adoption happens only after we inspect counts and document the change. No model results are generated.

## 1. Read the candidate manifest and human review

Edit `splits/pakistan/candidate_pair_review.csv` created by Notebook 03. Each `verdict` must be one of:

- `same_photo_or_adjacent_frame` — confirmed matching photograph, alternate encoding/crop, or adjacent video frame of the same physical car
- `likely_same_vehicle` — same physical vehicle reasonably apparent across related photographs
- `different` — different vehicle/photo despite similar model or dHash
- `uncertain` — unresolved; consult another group member before continuing

Mark reviewer initials and a brief reason as appropriate. Avoid using the publisher test set for model selection; this is solely a **leakage integrity audit**.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from hashlib import sha256
import json
import random

import pandas as pd
import yaml

cwd = Path.cwd().resolve()
REPO = next((p for p in (cwd, *cwd.parents) if (p / "configs" / "experiments.yaml").exists()), None)
if REPO is None:
    raise RuntimeError("Run this notebook inside the vehicle-recognition repository.")

CONFIG = yaml.safe_load((REPO / "configs/experiments.yaml").read_text(encoding="utf-8"))
DATA = CONFIG["datasets"]["pakistan"]
SEED = int(CONFIG["random_seed"])
VAL_FRACTION = float(DATA["split"]["validation_fraction_of_publisher_train"])
DIR = REPO / DATA["split_manifest_dir"]

manifest = pd.read_csv(DIR / "candidate_manifest.csv", dtype={"relative_path": str})
review = pd.read_csv(DIR / "candidate_pair_review.csv").fillna("")
class_map = json.loads((DIR / "candidate_class_to_idx.json").read_text(encoding="utf-8"))
metadata = json.loads((DIR / "candidate_split_metadata.json").read_text(encoding="utf-8"))

assert metadata["status"] == "CANDIDATE_NOT_APPROVED"
assert sha256((DIR / "candidate_manifest.csv").read_bytes()).hexdigest() == metadata["manifest_sha256"], "Candidate manifest changed since generation."
assert len(manifest) == 3847 and manifest["class_name"].nunique() == 48
assert not manifest["relative_path"].duplicated().any()
assert len(review) == len(set(review["review_id"])), "Duplicate review IDs."
assert len(review) == (metadata["publisher_train_vs_test_candidate_pairs"] + metadata["candidate_train_vs_val_candidate_pairs"]), "Review CSV does not represent all flagged pairs. Regenerate or investigate."

allowed = {"same_photo_or_adjacent_frame", "likely_same_vehicle", "different"}
unresolved = review.loc[~review["verdict"].isin(allowed)]
if len(unresolved):
    print("Unresolved review rows:", len(unresolved))
    display(unresolved[["review_id","verdict","left_path","right_path"]].head(15))
    raise RuntimeError("Complete the visual review before producing a cleaned candidate split.")

paths = set(manifest["relative_path"])
if not set(review["left_path"]).issubset(paths) or not set(review["right_path"]).issubset(paths):
    raise RuntimeError("Review references files not in the candidate manifest.")
print("Reviewed pairs:", len(review))
print("Verdicts:")
display(review["verdict"].value_counts().to_frame("pairs"))
print("Candidate manifest checksum:", metadata["manifest_sha256"])

Reviewed pairs: 35
Verdicts:


,pairs
verdict,
same_photo_or_adjacent_frame,35


Candidate manifest checksum: 97fa106c50e35fd184ea6ee8535e345599555f9af50c7ee6e2e3863b5da1ca82


## 2. Build groups from confirmed overlap edges

We use a **union–find graph**: each image is a node; a reviewed same-image/same-physical-vehicle match joins two nodes. Transitively connected images form one group. All members of a confirmed group must stay on the same side of train/validation, unless the group touches publisher test—in which case original publisher train members will be excluded.

If a reviewer marked a confirmed match across two *different class labels*, stop to resolve the possible labeling error instead of automatically merging them.

In [2]:
parent = {p: p for p in manifest["relative_path"]}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

labels = dict(zip(manifest["relative_path"], manifest["class_name"]))
confirmed = review[review["verdict"].isin(["same_photo_or_adjacent_frame", "likely_same_vehicle"])]
for r in confirmed.itertuples(index=False):
    if labels[r.left_path] != labels[r.right_path]:
        raise RuntimeError(f"Confirmed pair {r.review_id} crosses class labels. Verify labels before proceeding.")
    union(r.left_path, r.right_path)

members = defaultdict(list)
for p in sorted(paths):
    members[find(p)].append(p)

origin = dict(zip(manifest["relative_path"], manifest["publisher_split"]))
test_linked = {root for root, nodes in members.items() if any(origin[n] == "test" for n in nodes)}
excluded = sorted(n for root in test_linked for n in members[root] if origin[n] == "train")

print("Confirmed overlap pair edges:", len(confirmed))
print("Connected groups with publisher test:", len(test_linked))
print("Distinct publisher-training images to exclude (not pair count):", len(excluded))
print("Publisher test images retained:", (manifest["publisher_split"] == "test").sum())
assert all(origin[p] == "train" for p in excluded)

Confirmed overlap pair edges: 35
Connected groups with publisher test: 751
Distinct publisher-training images to exclude (not pair count): 23
Publisher test images retained: 751


## 3. Deterministic approximate stratified group split

Among the *remaining* publisher-training images, select entire connected groups for validation, separately within each of 48 vehicle classes. This keeps all confirmed connected matches together. It aims for about 15% validation **per class**, so exact overall counts can differ from the original 2,631/465/751 split.

The pseudorandom shuffle uses seed 42 and stable sorted paths, and we save explicit output assignments. It is a practical group-aware approximation, **not** a guarantee that unknown video-frame duplicates do not remain.

No experiment should begin until all classes have both training and validation examples.

In [3]:
eligible = manifest[(manifest["publisher_split"] == "train") & ~manifest["relative_path"].isin(excluded)]
eligible_paths = set(eligible["relative_path"])
grouped_by_class = defaultdict(list)
for root, nodes in sorted(members.items()):
    keep = [n for n in nodes if n in eligible_paths]
    if not keep:
        continue
    if len(keep) != len(nodes):
        raise RuntimeError("Unexpected partly retained group; check test-linked exclusions.")
    grouped_by_class[labels[keep[0]]].append(tuple(sorted(keep)))

rng = random.Random(SEED)
val_paths = set()
for cls in sorted(class_map):
    groups = sorted(grouped_by_class[cls])
    total = sum(len(g) for g in groups)
    if total < 2 or len(groups) < 2:
        raise RuntimeError(f"Not enough independent groups for train/val in {cls}: {total} images, {len(groups)} groups.")
    target = max(1, min(total - 1, round(VAL_FRACTION * total)))
    rng.shuffle(groups)

    # Deterministically choose groups that bring the validation count closer to target.
    chosen_indices = set()
    selected = 0
    while True:
        remaining = [(i,g) for i,g in enumerate(groups) if i not in chosen_indices]
        options = [(i,g) for i,g in remaining if selected + len(g) < total]
        if not options:
            break
        improvement = sorted(
            ((abs(target-selected)-abs(target-(selected+len(g))), i,g) for i,g in options),
            key=lambda x: (-x[0], x[1])
        )
        best_gain, idx, g = improvement[0]
        if selected > 0 and best_gain <= 0:
            break
        chosen_indices.add(idx)
        selected += len(g)
        if selected >= target:
            break
    if not chosen_indices:
        raise RuntimeError(f"No validation group selected for {cls}.")
    for i in chosen_indices:
        val_paths.update(groups[i])

def assign(row):
    p = row["relative_path"]
    if row["publisher_split"] == "test":
        return "test"
    if p in excluded:
        return "excluded_test_overlap"
    return "val" if p in val_paths else "train"

clean = manifest.copy()
clean["split"] = clean.apply(assign, axis=1)
clean = clean.sort_values(["split","class_idx","relative_path"]).reset_index(drop=True)
active = clean[clean["split"].isin(["train","val","test"])]
counts = active.groupby("split").size().to_dict()
coverage = active.groupby("split")["class_name"].nunique().to_dict()
per_class = pd.crosstab(active["class_name"], active["split"])
assert coverage == {"test":48, "train":48, "val":48}, coverage
assert len(clean) == 3847
assert counts["test"] == 751
assert clean["relative_path"].is_unique
assert set(clean.loc[clean["split"].eq("test"),"relative_path"]) == set(manifest.loc[manifest["publisher_split"].eq("test"),"relative_path"])
for nodes in members.values():
    occupied = set(clean.loc[clean["relative_path"].isin(nodes),"split"])
    assert len(occupied - {"excluded_test_overlap"}) <= 1, (nodes,occupied)
assert not any(p in val_paths for p in excluded)

print("Cleaned candidate counts:", clean["split"].value_counts().to_dict())
print("Class coverage:", coverage)
print("Minimum active examples/class:", per_class.min().to_dict())
print("Publisher test unchanged: 751 images")
display(per_class.sort_values("train").head(12))

Cleaned candidate counts: {'train': 2610, 'test': 751, 'val': 463, 'excluded_test_overlap': 23}
Class coverage: {'test': 48, 'train': 48, 'val': 48}
Minimum active examples/class: {'test': 3, 'train': 11, 'val': 2}
Publisher test unchanged: 751 images


split,test,train,val
class_name,,,
Suzuki_carry,3,11,2
Toyota_Landcruser,4,14,3
Suzuki_margala,4,14,2
Honda_civic_1994,4,15,3
Toyota_premio,4,15,3
Suzuki_Every,5,17,3
Toyota_Prado,5,17,3
Toyota_axio,4,17,3
Honda_Grace,5,18,3


## 4. Export the **cleaned candidate** manifest and audit trail

We keep the excluded image paths in the manifest with `split=excluded_test_overlap`; they are **not loaded** by training or validation. That is intentional: it preserves a trace of every original image.

Do not replace `candidate_manifest.csv` or raw data. No images are physically deleted.

The training pipeline will eventually load only rows labeled `train`, `val`, and `test` from the **approved final** manifest, with a frozen class-to-index mapping shared between both networks.

In [4]:
out_csv = DIR / "candidate_clean_manifest.csv"
clean.to_csv(out_csv, index=False)
excluded_path = DIR / "candidate_excluded_test_overlaps.csv"
clean.loc[clean["split"].eq("excluded_test_overlap"), ["relative_path", "publisher_split", "class_name", "class_idx"]].to_csv(excluded_path, index=False)
info = {
    "status":"CLEAN_CANDIDATE_AWAITING_APPROVAL",
    "policy":"retain all publisher test; exclude original training members of confirmed test-linked overlap components; group remaining confirmed overlaps before split",
    "source_manifest_sha256":metadata["manifest_sha256"],
    "clean_manifest_sha256":sha256(out_csv.read_bytes()).hexdigest(),
    "seed":SEED,
    "validation_fraction_target":VAL_FRACTION,
    "confirmed_pairs":len(confirmed),
    "reviewed_pair_count":len(review),
    "excluded_distinct_publisher_training_images":len(excluded),
    "counts":{k:int(v) for k,v in clean["split"].value_counts().to_dict().items()},
    "class_coverage":{k:int(v) for k,v in coverage.items()},
    "limitation":"dHash screen and human review cannot exclude all possible visually related frames",
}
(DIR / "candidate_clean_metadata.json").write_text(json.dumps(info,indent=2)+"\n",encoding="utf-8")
print("Cleaned manifest:",out_csv)
print("Manifest SHA-256:",info["clean_manifest_sha256"])
print("Excluded paths file:",excluded_path)
print("STATUS: CLEAN CANDIDATE — review outputs and approve before training.")

Cleaned manifest: /Users/bata/Documents/vehicle-make-model-recognition/splits/pakistan/candidate_clean_manifest.csv
Manifest SHA-256: 11ebb90a7a4530a2707f4d086131f93f4b3c91cae013bc6add0bc2417bff0f32
Excluded paths file: /Users/bata/Documents/vehicle-make-model-recognition/splits/pakistan/candidate_excluded_test_overlaps.csv
STATUS: CLEAN CANDIDATE — review outputs and approve before training.


## Share this checkpoint with Project HQ

Report **(1)** verdict counts, **(2)** number of *distinct* excluded publisher-training images, **(3)** final candidate train/val/test counts, **(4)** minimum active images per class, and **(5)** the cleaned manifest SHA-256.

Do not assume 23 flagged publisher train/test pairs means exactly 23 images excluded. One image can belong to several pairs or overlap groups.

Once reviewed, freeze an approved manifest, commit small split CSV/JSON artifacts and use them for **both Xception and ConvNeXt**. The original publisher test split remains available for a clearly disclosed, contamination-screened evaluation; report that cleaning changed the training pool relative to the publisher's original 3,096 images.